In [4]:
env = "local".lower()

if env == "local":
    data_path = "./dataset"
    result_path = "./results"
elif env == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    data_path = "/content/drive/MyDrive/Coding/dataset"
    result_path = "/content/drive/MyDrive/Coding/results"

# Data processing

In [5]:
import os
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
# from matplotlib.colors import Normalize
# from metpy.plots import ctables

In [6]:
# Load image names and label from metadata
metadata = pd.read_csv(f"{data_path}/metadata.csv", encoding="utf-8").set_index('id')

datetimes = metadata['start_datetime']
datetimes = pd.to_datetime(datetimes, format="%Y-%m-%d %H:%M:%S")

images = datetimes.dt.strftime('%Y%m%d_%H%M%S').tolist()
labels = metadata['tags'].fillna("").tolist()

data = pd.DataFrame({'images': images, 'labels': labels})
print(data)

               images     labels
0     20101101_000000  rain snow
1     20101101_135000  rain snow
2     20101102_000000  rain snow
3     20101107_000000  rain snow
4     20101107_205500  rain snow
...               ...        ...
1727  20191028_191500       rain
1728  20191029_000000       rain
1729  20191030_000000       rain
1730  20191030_024000       rain
1731  20191031_000000           

[1732 rows x 2 columns]


In [27]:
# Generate images from hdf5 files
temp_images = images.copy()
filenames = sorted(os.listdir(f'{data_path}/hdf5'))
with tqdm(total=len(filenames)) as progress:
    for filename in filenames:
        if filename == "all_data.hdf5":
            progress.update(1)
            continue
        with h5py.File(f'{data_path}/hdf5/{filename}', "r") as file:
            for key in list(file.keys()):
                obj = file[key][()][0]
                obj[obj < 5] = 0
                save_name = temp_images.pop(0)
                plt.imshow(obj, cmap='gist_ncar', interpolation='nearest')
                plt.axis('off')
                plt.savefig(f'{data_path}/image/{save_name}.png', bbox_inches='tight', pad_inches=0)
                plt.clf()
                progress.update(1)

100%|██████████| 1400/1400 [00:00<00:00, 1411.38it/s]


In [14]:
# Split the data into training and testing sets
seed = 42 # Seed for reproducibility
train_data = data.sample(frac=0.8,random_state=seed)
test_data = data.drop(train_data.index).sample(frac=1.0, random_state=seed)

train_data.to_csv(f'{data_path}/train_data.csv', index=False)
test_data.to_csv(f'{data_path}/test_data.csv', index=False)